# Модель

In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score

# Загрузим данные
df = pd.read_csv(r'C:\Users\nikka\OneDrive\Рабочий стол\ОТП Задания\ML\train.csv')
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'])

In [2]:
df.head()

,customer_id,snapshot_date,dpd_bucket,outstanding_amount,income,region,product_type,contacts_last_7d,rpc_last_30d,promises_last_30d,payments_amount_last_30d,target_paid_30d
0,100056,2025-10-01,1-30,10987.42,NaN,Siberia,micro_loan,3,0,1,0.00,0
1,100110,2025-10-01,61-90,30961.88,75940.84,Ural,installment,3,1,0,0.00,0
2,100111,2025-10-01,31-60,29000.38,82695.80,Center,installment,3,2,1,4746.34,1
3,100130,2025-10-01,1-30,21187.05,107453.10,Center,micro_loan,1,1,1,4280.34,1
4,100285,2025-10-01,1-30,33901.72,58239.00,South,micro_loan,1,0,0,2275.41,0


In [3]:
# Оценим баланс классов
df['target_paid_30d'].value_counts(normalize=True)

target_paid_30d
0    0.613088
1    0.386912
Name: proportion, dtype: float64

Как мы видим, сильного дисбаланса в классах нет

In [5]:
# Разделим данные по времени
train_mask = (df['snapshot_date'] >= '2025-10-01') & (df['snapshot_date'] <= '2025-12-31')
val_mask = (df['snapshot_date'] >= '2026-01-01') & (df['snapshot_date'] <= '2026-01-31')

df_train = df.loc[train_mask].copy()
df_val = df.loc[val_mask].copy()

# Оценим размеры тренировочной и валидационной выборки
print(f'Размер тренировочной выборки: {df_train.shape}')
print(f'Размер валидационной выборки: {df_val.shape}')

Размер тренировочной выборки: (5059, 12)
Размер валидационной выборки: (1741, 12)


In [6]:
# Разобъем данные на фичи и таргет
target_col = 'target_paid_30d'
drop_cols = ['customer_id', 'snapshot_date', target_col]

X_train = df_train.drop(columns=drop_cols)
y_train = df_train[target_col]

X_val = df_val.drop(columns=drop_cols)
y_val = df_val[target_col]

# Разобъем признаки на числовые и категориальные
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_features = X_train.select_dtypes(include=[np.number, 'bool']).columns.tolist()

print('Категориальные признаки:', categorical_features)
print('Числовые признаки:', numeric_features)

Категориальные признаки: ['dpd_bucket', 'region', 'product_type']
Числовые признаки: ['outstanding_amount', 'income', 'contacts_last_7d', 'rpc_last_30d', 'promises_last_30d', 'payments_amount_last_30d']


In [7]:
# Построим пайплайн для предобработки данных
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

In [8]:
# Построим baseline-модель с использованием логистической регрессии
# Переберем две опции - с парамаетром 'balanced' и без

results = []

for class_weight_value in [None, 'balanced']:
    # Посторим пайплайн для создания модели
    model = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('classifier', LogisticRegression(max_iter=1000, class_weight=class_weight_value, random_state=42)),
        ]
    )

    # Обучим модель
    model.fit(X_train, y_train)

    # Сделаем предсказание
    val_pred_proba = model.predict_proba(X_val)[:, 1]

    # Посчитаем метрики ROC-AUC и PR-AUC
    roc_auc = roc_auc_score(y_val, val_pred_proba)
    pr_auc = average_precision_score(y_val, val_pred_proba)

    # Вычислим метрику precision@top30%
    val_scored = df_val.copy()
    val_scored['pred_proba'] = val_pred_proba
    top_k = int(np.ceil(len(val_scored) * 0.30))
    val_top30 = val_scored.sort_values('pred_proba', ascending=False).head(top_k)
    precision_top30 = val_top30[target_col].mean()

    # Соберем результаты работы моделей в одну таблицу
    results.append({'class_weight': class_weight_value, 'roc_auc': roc_auc, 'pr_auc': pr_auc, 'precision_top30': precision_top30})

In [9]:
# Сравним результаты работы моделей
results_df = pd.DataFrame(results)

print('\nМетрики на валидационной выборке:')
print(results_df)


Метрики на валидационной выборке:
  class_weight   roc_auc    pr_auc  precision_top30
0         None  0.789407  0.719074         0.701721
1     balanced  0.789608  0.719377         0.701721


Как мы видим по метрикам ROC-AUC и PR-AUC - модель с параметром 'balanced' выдает более высокие результаты.

В дальнейшем, мы можем это подтвердить или опровергнуть с помощью статистического теста.

In [10]:
# Оценим, насколько модель делает предсказания лучше, чем случайное предсказание
base_rate = y_val.mean()
print(f'\nМетрика предсказания случайным значением на валидационной выборке: {base_rate:.4f}')

uplift = precision_top30 / base_rate
print(f'По метрике precision@top-30% baseline-модель выдает результат выше, чем случайное предсказание в {uplift:.2f} раз')


Метрика предсказания случайным значением на валидационной выборке: 0.3889
По метрике precision@top-30% baseline-модель выдает результат выше, чем случайное предсказание в 1.80 раз


# Вывод

Baseline-модель логистической регрессии показывает хорошее качество ранжирования: ROC-AUC = 0.79 и PR-AUC = 0.72 на валидационной выборке. 

Бизнес-метрика precision@top30% составляет около 0.70, что существенно выше base rate (0.39) и даёт uplift около 1.8x по сравнению со случайным отбором. 

Это означает, что модель эффективно концентрирует платящих клиентов в верхнем сегменте скоринга и может использоваться для приоритизации. 

Использование class_weight='balanced' может повлиять на качество модели, однако это нужно доказать отдельно с помощью статистического теста.